# Lab 04: Your First Chain (LCEL) -- Solution

**Goal:** Build your first chain using LCEL pipe syntax: `prompt | llm | parser`

**What you'll learn:**
- What LCEL (LangChain Expression Language) is
- The pipe operator `|` connects components
- `StrOutputParser` extracts text from the response
- Why chains are better than manual step-by-step calls

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="llama3.2:1b")

## Step 1: The Manual Way (Without Chains)

This works, but it's verbose and hard to reuse.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("human", "{question}"),
])

messages = prompt.invoke({"question": "What is Flask?"})
response = llm.invoke(messages)
text = response.content
print(text)

## Step 2: The LCEL Way (One Chain, One Line)

The pipe `|` operator connects: prompt -> llm -> parser.
Each component's output automatically becomes the next one's input.

In [ ]:
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"question": "What is Flask?"})
print(result)
print(f"Type: {type(result)}")

## Step 3: Understand the Data Flow

Let's trace what happens at each step.

In [ ]:
stage1 = prompt.invoke({"question": "What is FastAPI?"})
print(f"After prompt:  {type(stage1).__name__} -> {len(stage1.to_messages())} messages")

stage2 = llm.invoke(stage1)
print(f"After LLM:     {type(stage2).__name__} -> '{stage2.content[:50]}...'")

parser = StrOutputParser()
stage3 = parser.invoke(stage2)
print(f"After parser:  {type(stage3).__name__} -> '{stage3[:50]}...'")

## Step 4: Reuse the Chain with Different Inputs

The chain is a reusable object. Pass different inputs each time.

In [ ]:
questions = [
    "What is Git in one sentence?",
    "What is an API in one sentence?",
    "What is a database in one sentence?",
]
for q in questions:
    answer = chain.invoke({"question": q})
    print(f"Q: {q}")
    print(f"A: {answer}\n")

## TODO 1: Build a Chain That Translates Text

Create a chain: `translation_prompt | llm | StrOutputParser()`

The prompt should accept `{text}` and `{language}` variables.

In [ ]:
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Return only the translation."),
    ("human", "Translate '{text}' to {language}"),
])
translate_chain = translate_prompt | llm | StrOutputParser()

result = translate_chain.invoke({"text": "Welcome to India", "language": "Tamil"})
print(f"Translation: {result}")

## TODO 2: Build a Chain That Summarizes Text

Create a chain that takes `{text}` and `{max_words}` as input
and returns a summary within the word limit.

In [ ]:
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the given text in {max_words} words or fewer. Return only the summary."),
    ("human", "{text}"),
])
summarize_chain = summarize_prompt | llm | StrOutputParser()

result = summarize_chain.invoke({
    "text": "Flask is a lightweight Python web framework that provides tools and libraries for building web applications. It follows a micro-framework design philosophy, offering simplicity and flexibility while allowing developers to add extensions as needed. Flask is widely used for developing RESTful APIs and web services.",
    "max_words": "20",
})
print(f"Summary: {result}")

## Key Takeaways

- LCEL chain: `prompt | llm | parser`
- The pipe `|` passes output of each step to the next
- `StrOutputParser` extracts plain text from `AIMessage`
- Chains are reusable objects -- invoke with different inputs